## Завдання 1
Побудуйте модель поліноміальної регресії для однієї ознаки на базі датасету California Housing:

* Завантажте датасет fetch_california_housing.
* Використовуйте тільки одну ознаку MedInc.
* Створіть поліноміальні ознаки ступеня 2 за допомогою PolynomialFeatures.
* Розділіть дані на навчальну і тестову вибірки (80/20, random_state=42).
* Навчіть модель і обчисліть R² на тестовій вибірці.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import pandas as pd

data = fetch_california_housing()
X = data.data[:, [0]]  # MedInc
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('lr', LinearRegression())
])
pipe.fit(X_train, y_train)

print(f'R² (test): {pipe.score(X_test, y_test):.4f}')

R² (test): 0.4633


## Завдання 2
Перевірте, як ступінь полінома впливає на якість моделі:

* Використовуйте ознаку MedInc з California Housing.
* Побудуйте моделі з PolynomialFeatures ступенів 1, 2 і 3.
* Для кожної моделі обчисліть R² на train і test.
* Запишіть результати в таблицю.

In [2]:
results = []
for deg in [1, 2, 3]:
    p = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('lr', LinearRegression())
    ])
    p.fit(X_train, y_train)
    results.append({
        'Ступінь полінома': deg,
        'R² train': round(p.score(X_train, y_train), 4),
        'R² test':  round(p.score(X_test,  y_test),  4)
    })

print(pd.DataFrame(results).to_string(index=False))

 Ступінь полінома  R² train  R² test
                1    0.4770   0.4589
                2    0.4816   0.4633
                3    0.4908   0.4671


## Завдання 3
Застосуйте Lasso-регуляризацію і порівняйте зі звичайною лінійною регресією:

* Використовуйте датасет California Housing.
* Візьміть ознаки MedInc, HouseAge, AveRooms, AveBedrms.
* Побудуйте дві моделі: LinearRegression та Lasso(alpha=0.1).
* Розділіть дані на train/test (80/20, random_state=42).
* Для кожної моделі обчисліть R² на тестовій вибірці.
* Порівняйте коефіцієнти: скільки з них стали рівними 0 у Lasso.

In [3]:
import numpy as np
from sklearn.linear_model import Lasso

feature_names = list(data.feature_names)
feat_idx = [feature_names.index(f) for f in ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms']]
X34 = data.data[:, feat_idx]

X34_train, X34_test, y34_train, y34_test = train_test_split(X34, y, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X34_train, y34_train)
lasso = Lasso(alpha=0.1).fit(X34_train, y34_train)

df3 = pd.DataFrame({
    'Ознака': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms'],
    'LinearRegression': lr.coef_.round(4),
    'Lasso(alpha=0.1)': lasso.coef_.round(4)
})
print(df3.to_string(index=False))
print(f'\nR² LinearRegression: {lr.score(X34_test, y34_test):.4f}')
print(f'R² Lasso:             {lasso.score(X34_test, y34_test):.4f}')
print(f'Нульових коефіцієнтів у Lasso: {np.sum(lasso.coef_ == 0)}')

   Ознака  LinearRegression  Lasso(alpha=0.1)
   MedInc            0.5453            0.4051
 HouseAge            0.0161            0.0163
 AveRooms           -0.2246           -0.0004
AveBedrms            1.1130            0.0000

R² LinearRegression: 0.5089
R² Lasso:             0.4933
Нульових коефіцієнтів у Lasso: 1


## Завдання 4
Застосуйте Ridge-регуляризацію і порівняйте зі звичайною лінійною регресією:

* Використовуйте ті самі ознаки (MedInc, HouseAge, AveRooms, AveBedrms).
* Побудуйте дві моделі: LinearRegression та Ridge(alpha=1.0).
* Розділіть дані на train/test (80/20, random_state=42).
* Оформіть результати в таблиці: Модель | Тест R² | Кількість ненульових коеф.

In [4]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0).fit(X34_train, y34_train)

df4 = pd.DataFrame({
    'Модель': ['LinearRegression', 'Ridge(alpha=1.0)'],
    'Тест R²': [
        round(lr.score(X34_test, y34_test), 4),
        round(ridge.score(X34_test, y34_test), 4)
    ],
    'Ненульових коеф.': [
        int(np.sum(lr.coef_ != 0)),
        int(np.sum(ridge.coef_ != 0))
    ]
})
print(df4.to_string(index=False))

          Модель  Тест R²  Ненульових коеф.
LinearRegression   0.5089                 4
Ridge(alpha=1.0)   0.5090                 4
